# ライブラリのインポート

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import lingam

# データの読み込み

In [ ]:
df = pd.read_excel("excelファイルの絶対パスを記入")
df = df.sort_values("date").reset_index(drop=True) #(日付順に並び直しインデックスをつけ直す)
df.head()

In [ ]:
# 使用するカラムを指定し、df_lingamを作成
use_columns = [
    "",
    "",
    "",
    ""
]
df_lingam = df[use_columns].copy()

# VAR-LiNGAMのモデル作成

In [ ]:
# モデル生成セル
model = lingam.VARLiNGAM(
    lags=lags,
    criterion=criterion,
    prune=prune,
    ar_coefs=ar_coefs,
    lingam_model=lingam_model,
    random_state=random_state
)

In [ ]:
# ラグの設定 (強制的にn期前までラグを指定する場合)
lags = n
criterion = None

In [ ]:
# ラグの設定(AIに次数を自動選択させる場合)
lags = n
criterion = "bic" #("aic","bic",hqic,"fpe"などから選択)

In [ ]:
# 不要な因果関係の刈りこみ
prune = True

In [ ]:
# 外部からVARの係数を与えない(与える場合は行列式を挿入)
ar_coefs = None

In [ ]:
# 同時点の因果構造を推定するときのLiNGAMモデルの指定
lingam_model = None

In [ ]:
# random_stateの設定
random_state = 42 # 好きな数字入れてください

# モデルの実行

In [ ]:
model.fit(df_lingam)

In [ ]:
# 採用されたラグ数の確認
print("採用されたラグ数:", model.lags_)

In [ ]:
# adjacency_matrices_の確認
print(model.adjacency_matrices_.shape)

In [ ]:
#　全ラグをdf化
for lag, matrix in enumerate(model.adjacency_matrices_):
    adj_df = pd.DataFrame(
        matrix,
        index=use_columns,
        columns=use_columns
    )

    print(f"===== lag {lag} =====")
    display(adj_df)

In [ ]:
# 係数が0以外のものだけ抽出して表示
edges = []

for lag, matrix in enumerate(model.adjacency_matrices_):
    for i, effect in enumerate(use_columns):
        for j, cause in enumerate(use_columns):
            coef = matrix[i, j]

            if coef != 0:
                edges.append([
                    lag,
                    cause,
                    effect,
                    coef
                ])
                
edges_df = pd.DataFrame(
    edges,
    columns=["lag", "cause", "effect", "coefficient"]
)

edges_df

In [ ]:
# 係数が小さすぎるものも除外
# 係数の閾値の設定
threshold = 0.1 #(任意の値を設定)

edges = []

for lag, matrix in enumerate(model.adjacency_matrices_):
    for i, effect in enumerate(use_columns):
        for j, cause in enumerate(use_columns):
            coef = matrix[i, j]

            if coef >= threshold:
                edges.append([
                    lag,
                    cause,
                    effect,
                    coef
                ])
                
edges_df = pd.DataFrame(
    edges,
    columns=["lag", "cause", "effect", "coefficient"]
)

edges_df

# 結果の可視化

In [ ]:
# 表示対象の限定
plot_edges = edges_df[
    edges_df["coefficient"].abs() >= threshold
].copy()

In [ ]:
# 確認
plot_edges

In [ ]:
# 時間軸込みのVAR-LiNGAM用DAGの組み立て
G = nx.DiGraph()

for _, row in plot_edges.iterrows():
    lag = int(row["lag"])

    if lag == 0:
        cause_node = f"{row['cause']}(t)"
    else:
        cause_node = f"{row['cause']}(t-{lag})"

    effect_node = f"{row['effect']}(t)"

    G.add_edge(
        cause_node,
        effect_node,
        weight=abs(row["coefficient"]),
        coefficient=row["coefficient"]
    )

In [ ]:
pos = {}

for node in G.nodes:
    if "(t-" in node:
        lag = int(node.split("(t-")[1].replace(")", ""))
        x = -lag
    else:
        x = 0

    variable = node.split("(")[0]
    y = use_columns.index(variable)

    pos[node] = (x, -y)

In [ ]:
plt.figure(figsize=(12, 7))

widths = [
    G[u][v]["weight"] * 3
    for u, v in G.edges
]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2500,
    font_size=10,
    arrowsize=20,
    width=widths
)

plt.show()

# モデル結果の安定性評価

In [ ]:
# ブートストラップの回数
n_sampling = 100

In [ ]:
# 実行セル
bootstrap_result = model.bootstrap(
    df_lingam,
    n_sampling=n_sampling
)

In [ ]:
probabilities = bootstrap_result.get_probabilities(
    min_causal_effect=threshold
)

for lag, matrix in enumerate(probabilities):
    prob_df = pd.DataFrame(
        matrix,
        index=use_columns,
        columns=use_columns
    )

    print(f"===== lag {lag} =====")
    display(prob_df)

In [ ]:
# 係数とブートストラップ確率を1枚の表にまとめる
edge_probabilities = []

for _, row in edges_df.iterrows():
    lag = int(row["lag"])

    cause_idx = use_columns.index(row["cause"])
    effect_idx = use_columns.index(row["effect"])

    probability = probabilities[lag][effect_idx, cause_idx]

    edge_probabilities.append(probability)
    
    edges_df["probability"] = edge_probabilities

edges_df

In [ ]:
# Probabilityが80%以上のものだけ選んでDAGとして出力
probability_threshold = 0.8

plot_edges = edges_df[
    edges_df["probability"] >= probability_threshold
].copy()

G = nx.DiGraph()

for _, row in plot_edges.iterrows():
    lag = int(row["lag"])

    if lag == 0:
        cause_node = f"{row['cause']}(t)"
    else:
        cause_node = f"{row['cause']}(t-{lag})"

    effect_node = f"{row['effect']}(t)"

    G.add_edge(
        cause_node,
        effect_node,
        weight=abs(row["coefficient"]),
        coefficient=row["coefficient"]
    )

# ノードの配置（過去 → 現在）
pos = {}

for node in G.nodes:
    if "(t-" in node:
        lag = int(node.split("(t-")[1].replace(")", ""))
        x = -lag
    else:
        x = 0

    variable = node.split("(")[0]
    y = use_columns.index(variable)

    pos[node] = (x, -y)

# DAGを描画
plt.figure(figsize=(12, 7))

widths = [
    G[u][v]["weight"] * 3
    for u, v in G.edges
]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2500,
    font_size=10,
    arrowsize=20,
    width=widths
)

plt.show()